# Title Page

**CSP Programming Assignment: The Zebra Puzzle**

**Course:** ICOM 5015 — Artificial Intelligence  

**Topic:** Constraint Satisfaction Problems  

**Assignment:** Exercise 7, Chapter 6  

**Prepared by:** Victor A. Gerena Hilerio,Jesus Y. Cabán Feliciano,Alejandro A. Roberts Quintana,Nollan N. Rivera Febus 

**Institution:** University of Puerto Rico at Mayagüez  

**Date:** May 8, 2026

This notebook presents the solution to the Zebra puzzle as a Constraint Satisfaction Problem (CSP). The report is organized using the formal report structure from the writing guidelines: abstract, introduction, theory, discussion/results, conclusion, future work, references, and appendix.


# Abstract

This project solves the Zebra logic puzzle by formulating it as a Constraint Satisfaction Problem. The puzzle has five houses, five colors, five nationalities, five candy brands, five drinks, and five pets. The objective is to determine where the zebra lives and which house drinks water. The implemented representation uses one CSP variable per attribute, with each variable assigned a house position from 1 to 5. The constraints represent the clues given in the problem, such as equality, adjacency, ordering, and all-different restrictions. The program uses the AIMA Python CSP framework with backtracking search, minimum-remaining-values variable selection, and forward checking. The resulting solution is that the **Japanese person owns the zebra**, and the **Norwegian drinks water**.


# Table of Contents

1. [Introduction](#1.-Introduction)  
2. [Theoretical Background](#2.-Theoretical-Background)  
3. [Discussion and Implementation](#3.-Discussion-and-Implementation)  
   3.1 [Problem Statement](#3.1-Problem-Statement)  
   3.2 [CSP Representation Used](#3.2-CSP-Representation-Used)  
   3.3 [Implementation](#3.3-Implementation)  
   3.4 [Solution and Verification](#3.4-Solution-and-Verification)  
   3.5 [Alternative CSP Representations](#3.5-Alternative-CSP-Representations)  
4. [Conclusion](#4.-Conclusion)  
5. [Future Work](#5.-Future-Work)  
6. [References](#6.-References)  


# List of Tables

**Table 1.** Final solved arrangement of the five houses.  
**Table 2.** Comparison of possible CSP representations for the Zebra puzzle.


# 1. Introduction

## Background

The Zebra puzzle is a classic logic puzzle that can be solved by reasoning through a list of clues. Each house has exactly one color, one nationality, one candy brand, one drink, and one pet. No two houses share the same value in any category. Because the problem is built from variables, domains, and restrictions, it fits naturally as a Constraint Satisfaction Problem.

## Purpose

The purpose of this assignment is to solve the puzzle programmatically and to discuss different ways the problem can be represented as a CSP. The final answer should identify the owner of the zebra and the person who drinks water.

## Scope

This notebook focuses on a complete and readable CSP solution. It uses the AIMA Python CSP implementation instead of writing a solver from scratch. The report also compares three possible representations and explains why one representation is preferred for this problem.


# 2. Theoretical Background

A Constraint Satisfaction Problem is defined by three main components: variables, domains, and constraints. Variables are the unknowns that must be assigned values. Domains contain the possible values for each variable. Constraints restrict which combinations of values are allowed.

For the Zebra puzzle, a state is not a sequence of actions. The important part is the final complete assignment that satisfies every clue. This makes the problem different from path-search problems such as route planning. In this assignment, the solver searches through possible assignments and rejects any partial assignment that violates a constraint.

The main solving method used here is backtracking search. Backtracking assigns values to variables one at a time and returns to an earlier decision when a contradiction is found. The implementation also uses the minimum-remaining-values heuristic, which tries to choose the most constrained unassigned variable first, and forward checking, which removes inconsistent values from neighboring domains after each assignment.


# 3. Discussion and Implementation

This section contains the actual CSP formulation and solution. The main representation used here assigns house numbers to attributes. For example, instead of saying “House 1 has the Norwegian,” the CSP says `Norwegian = 1`. This makes clues such as “The Norwegian lives next to the blue house” easy to write as `abs(Norwegian - Blue) == 1`.


## 3.1 Problem Statement

There are five houses numbered from left to right as 1 through 5. Each house has one color, one nationality, one candy, one drink, and one pet. The clues are translated into constraints. For example:

- “The Englishman lives in the red house” becomes `Englishman == Red`.
- “The green house is immediately to the right of the ivory house” becomes `Green == Ivory + 1`.
- “The Norwegian lives next to the blue house” becomes `abs(Norwegian - Blue) == 1`.
- “Milk is drunk in the middle house” becomes `Milk == 3`.

The two questions to answer are:

1. Where does the zebra live?
2. In which house do they drink water?


## 3.2 CSP Representation Used

The representation used in the code has one variable for every attribute:

- Colors: `Red`, `Green`, `Ivory`, `Yellow`, `Blue`
- Nationalities: `Englishman`, `Spaniard`, `Norwegian`, `Ukrainian`, `Japanese`
- Candies: `Hershey`, `KitKats`, `Smarties`, `Snickers`, `MilkyWays`
- Drinks: `Coffee`, `Tea`, `Milk`, `OrangeJuice`, `Water`
- Pets: `Dog`, `Fox`, `Snails`, `Horse`, `Zebra`

Each variable has domain `{1, 2, 3, 4, 5}`, representing the house position. The values in each category must be all different. For example, the five colors must occupy five different houses.

This representation is compact because the solver only has 25 variables, and most clues become simple binary constraints between two variables.


## 3.3 Implementation

The following code loads the AIMA Python CSP tools, defines the Zebra puzzle variables, domains, and constraints, and solves the problem using backtracking search.


In [ ]:
import sys
from pathlib import Path

# ------------------------------------------------------------
# AIMA Python setup
# ------------------------------------------------------------
# Place the extracted AIMA Python folder in the same directory as this notebook.
# Expected structure:
#
#   CSP_Zebra_Puzzle_Formal_Report.ipynb
#   aima-python/
#       csp.py
#       search.py
#       utils.py
#       ...
#
# If the folder has a slightly different name, the code also checks for
# common alternatives such as "aima-python-master".

candidate_dirs = [
    Path.cwd() / "aima-python",
    Path("aima-python"),
]

AIMA_DIR = next((folder for folder in candidate_dirs if (folder / "csp.py").exists()), None)

if AIMA_DIR is None:
    raise FileNotFoundError(
        "Could not find the extracted AIMA Python folder. "
        "Put the folder named 'aima-python' beside this notebook and rerun this cell."
    )

sys.path.insert(0, str(AIMA_DIR.resolve()))

from csp import CSP, backtracking_search, mrv, forward_checking

print(f"AIMA Python CSP tools loaded successfully from: {AIMA_DIR.resolve()}")


AIMA Python CSP tools loaded successfully from: C:\Users\Jesus Y\Downloads\AI_Assigment5\aima-python


In [2]:
# ------------------------------------------------------------
# Variables and domains
# ------------------------------------------------------------

houses = range(1, 6)  # 1 is the leftmost house; 5 is the rightmost house.

colors = ["Red", "Green", "Ivory", "Yellow", "Blue"]
nations = ["Englishman", "Spaniard", "Norwegian", "Ukrainian", "Japanese"]
candies = ["Hershey", "KitKats", "Smarties", "Snickers", "MilkyWays"]
drinks = ["Coffee", "Tea", "Milk", "OrangeJuice", "Water"]
pets = ["Dog", "Fox", "Snails", "Horse", "Zebra"]

categories = [colors, nations, candies, drinks, pets]
variables = colors + nations + candies + drinks + pets

domains = {var: list(houses) for var in variables}

# Fixed-position clues can be represented directly by reducing the domain.
domains["Norwegian"] = [1]
domains["Milk"] = [3]


In [ ]:
# ------------------------------------------------------------
# Constraint construction helpers
# ------------------------------------------------------------

relations = {}

def add_constraint(a, b, relation):
    """Add a binary relation in both directions."""
    relations[(a, b)] = relation
    relations[(b, a)] = lambda y, x, relation=relation: relation(x, y)


def add_equality(a, b):
    add_constraint(a, b, lambda x, y: x == y)


def add_next_to(a, b):
    add_constraint(a, b, lambda x, y: abs(x - y) == 1)


def add_immediately_right(right, left):
    """The first variable is immediately to the right of the second."""
    add_constraint(right, left, lambda r, l: r == l + 1)



for category in categories:
    for i in range(len(category)):
        for j in range(i + 1, len(category)):
            add_constraint(category[i], category[j], lambda x, y: x != y)

# Puzzle clues.
add_equality("Englishman", "Red")
add_equality("Spaniard", "Dog")
add_immediately_right("Green", "Ivory")
add_next_to("Hershey", "Fox")
add_equality("KitKats", "Yellow")
add_next_to("Norwegian", "Blue")
add_equality("Smarties", "Snails")
add_equality("Snickers", "OrangeJuice")
add_equality("Ukrainian", "Tea")
add_equality("Japanese", "MilkyWays")
add_next_to("KitKats", "Horse")
add_equality("Coffee", "Green")



neighbors = {var: [] for var in variables}
for a, b in relations:
    if b not in neighbors[a]:
        neighbors[a].append(b)


def zebra_constraints(A, a, B, b):
    """AIMA CSP constraint callback."""
    return relations.get((A, B), lambda x, y: True)(a, b)

zebra_csp = CSP(variables, domains, neighbors, zebra_constraints)
print(f"Variables: {len(variables)}")
print(f"Binary constraint links: {len(relations)}")


Variables: 25
Binary constraint links: 122


In [ ]:
# ------------------------------------------------------------
# Solves the CSP
# ------------------------------------------------------------

solution = backtracking_search(
    zebra_csp,
    select_unassigned_variable=mrv,
    inference=forward_checking,
)

solution


{'Milk': 3,
 'Norwegian': 1,
 'Blue': 2,
 'Red': 3,
 'Englishman': 3,
 'Yellow': 1,
 'KitKats': 1,
 'Horse': 2,
 'Green': 5,
 'Coffee': 5,
 'Ivory': 4,
 'OrangeJuice': 4,
 'Snickers': 4,
 'Water': 1,
 'Tea': 2,
 'Ukrainian': 2,
 'Spaniard': 4,
 'Dog': 4,
 'Japanese': 5,
 'MilkyWays': 5,
 'Hershey': 2,
 'Smarties': 3,
 'Snails': 3,
 'Fox': 1,
 'Zebra': 5}

## 3.4 Solution and Verification

The next cell converts the assignment into a house-by-house table. This is easier to read than the raw dictionary returned by the solver.

**Table 1. Final solved arrangement of the five houses.**


In [5]:
import pandas as pd

rows = []
for house in houses:
    rows.append({
        "House": house,
        "Color": next(v for v in colors if solution[v] == house),
        "Nationality": next(v for v in nations if solution[v] == house),
        "Candy": next(v for v in candies if solution[v] == house),
        "Drink": next(v for v in drinks if solution[v] == house),
        "Pet": next(v for v in pets if solution[v] == house),
    })

solution_table = pd.DataFrame(rows)
solution_table


,House,Color,Nationality,Candy,Drink,Pet
0,1,Yellow,Norwegian,KitKats,Water,Fox
1,2,Blue,Ukrainian,Hershey,Tea,Horse
2,3,Red,Englishman,Smarties,Milk,Snails
3,4,Ivory,Spaniard,Snickers,OrangeJuice,Dog
4,5,Green,Japanese,MilkyWays,Coffee,Zebra


In [6]:
zebra_house = solution["Zebra"]
water_house = solution["Water"]

zebra_owner = next(n for n in nations if solution[n] == zebra_house)
water_drinker = next(n for n in nations if solution[n] == water_house)

print(f"The zebra lives in house {zebra_house}, owned by the {zebra_owner}.")
print(f"Water is drunk in house {water_house}, by the {water_drinker}.")


The zebra lives in house 5, owned by the Japanese.
Water is drunk in house 1, by the Norwegian.


In [7]:
# ------------------------------------------------------------
# Basic verification of all clues
# ------------------------------------------------------------

checks = {
    "Englishman in red house": solution["Englishman"] == solution["Red"],
    "Spaniard owns dog": solution["Spaniard"] == solution["Dog"],
    "Norwegian in first house": solution["Norwegian"] == 1,
    "Green immediately right of ivory": solution["Green"] == solution["Ivory"] + 1,
    "Hershey next to fox": abs(solution["Hershey"] - solution["Fox"]) == 1,
    "KitKats in yellow house": solution["KitKats"] == solution["Yellow"],
    "Norwegian next to blue": abs(solution["Norwegian"] - solution["Blue"]) == 1,
    "Smarties eater owns snails": solution["Smarties"] == solution["Snails"],
    "Snickers eater drinks orange juice": solution["Snickers"] == solution["OrangeJuice"],
    "Ukrainian drinks tea": solution["Ukrainian"] == solution["Tea"],
    "Japanese eats Milky Ways": solution["Japanese"] == solution["MilkyWays"],
    "KitKats next to horse": abs(solution["KitKats"] - solution["Horse"]) == 1,
    "Coffee in green house": solution["Coffee"] == solution["Green"],
    "Milk in middle house": solution["Milk"] == 3,
}

verification_table = pd.DataFrame([
    {"Clue": clue, "Satisfied": ok}
    for clue, ok in checks.items()
])

assert all(checks.values())
verification_table


,Clue,Satisfied
0,Englishman in red house,True
1,Spaniard owns dog,True
2,Norwegian in first house,True
3,Green immediately right of ivory,True
4,Hershey next to fox,True
5,KitKats in yellow house,True
6,Norwegian next to blue,True
7,Smarties eater owns snails,True
8,Snickers eater drinks orange juice,True
9,Ukrainian drinks tea,True


The table shows that all clues are satisfied. The final answers are:

- **The zebra is in House 5. The Japanese person owns the zebra.**
- **Water is drunk in House 1. The Norwegian drinks water.**


## 3.5 Alternative CSP Representations

There is more than one reasonable way to represent this puzzle as a CSP. The choice of representation matters because it changes how easy the constraints are to write and how efficiently the solver can search.

**Table 2. Comparison of possible CSP representations for the Zebra puzzle.**

| Representation | Variables | Domain | Advantages | Disadvantages |
|---|---|---|---|---|
| Attribute-as-variable | Each attribute is a variable, such as `Norwegian`, `Red`, or `Zebra` | House positions 1–5 | Compact, easy to express equality and adjacency clues, works well with binary CSP solvers | Requires all-different constraints inside each category |
| House-as-variable | Each house is a variable | Full tuples such as `(color, nationality, candy, drink, pet)` | Very close to how humans describe the puzzle | Domain is much larger because each house can take many combinations |
| Boolean/exact-cover representation | Variables such as `Color(Red, House3)` | True/False | Very systematic and suitable for SAT/exact-cover solvers | Many more variables and constraints; less readable for this small puzzle |

The preferred representation for this assignment is the **attribute-as-variable representation**. It keeps the number of variables small and turns most clues into direct binary relations. For example, adjacency constraints are simple arithmetic expressions over house numbers. This representation is also a good match for AIMA Python's CSP class, which expects variables, domains, neighbors, and binary constraints.


# 4. Conclusion

The Zebra puzzle was successfully solved as a Constraint Satisfaction Problem. The CSP used 25 variables, each representing the house position of one attribute. The solution satisfies every clue and gives a unique arrangement of the houses.

The final result is that the **Japanese person owns the zebra**, and the **Norwegian drinks water**. The representation chosen for the implementation was the attribute-as-variable representation because it is compact, readable, and works naturally with binary constraints. Other representations are possible, but they either increase the domain size or make the constraint set less direct.


# 5. Future Work

A possible extension would be to compare solver performance across different representations. For example, the same puzzle could be solved using a house-as-variable representation, a Boolean SAT representation, and the attribute-as-variable representation used here. The number of variables, number of constraints, and execution time could then be compared. Another improvement would be to add a small visualization of the five houses after the solution is found.


# 6. References

[1] N. G. Santiago and M. A. Jiménez, *Writing Formal Reports: An Approach for Engineering Students in 21st Century*, 3rd ed., Department of Electrical and Computer Engineering, University of Puerto Rico at Mayagüez, 2002.

[2] S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. Pearson, 2021.

[3] AIMA Python Project, “aima-python,” GitHub repository, accessed through the course-provided extracted `aima-python` folder.
